# Instruction Data and the Three Masks

Build shifted targets that score responses and EOS without learning from padding.

## Before You Start

Course 10. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** scratch-04_classification.

## What You Will Be Able to Do

- Collate variable-length prompt-response examples
- Distinguish causal, padding, and loss masks
- Run response-only supervised updates

## Prompt tokens are context even when they are not targets

For each example concatenate prompt IDs, response IDs, and one EOS ID. Construct input=sequence[:-1] and target=sequence[1:]. At input position i, the target is original token i+1. Score it when i+1 is at least the prompt length, so the final prompt state learns to predict the first response token. Include EOS in the response loss; otherwise the model receives no direct lesson about stopping.

Our collate_sft accepts nonempty prompt lists, response lists (possibly empty), eos, pad, and max_length. Each example must fit completely; raise rather than silently truncating away its response. Pad inputs and targets to the largest shifted length in the batch, return input attention masks and response loss masks, and leave padding unscored. An empty response still has an EOS target.

## Three masks answer three different questions

The causal mask prevents attention to future positions. The attention/padding mask prevents attending to absent input tokens. The loss mask selects which target errors contribute gradients. Zeroing prompt loss does not remove prompt tokens from attention: the answer should depend on the instruction. Packed multi-example training additionally needs document-boundary attention isolation; this lab uses one example per row instead.

Teacher forcing exposes the gold prefix during training, whereas generation consumes the model's own predictions. Evaluate both held-out response NLL and free-running exact-match or task checks. Keep near-duplicate prompts and answers from crossing splits. A decrease in training loss alone neither proves instruction following nor excludes memorization.

## Worked Example: Shift before deciding what to score

Prompt=[10,11], response=[20,21], EOS=2.

1. **Concatenate:** Sequence=[10,11,20,21,2].
2. **Shift:** Inputs=[10,11,20,21], targets=[11,20,21,2].
3. **Mask loss:** Loss mask=[0,1,1,1]. Position 1 predicts the first response token 20. All four input positions are real.

An off-by-one mask can remove the first response target or train on the prompt by accident.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Response-only SFT collation. Lists are token IDs, not raw text."""
import numpy as np

def collate_sft(examples, eos, pad, max_length):
    """Return dict input_ids, targets, attention_mask, loss_mask as [B,T] arrays.

examples is a nonempty list of (nonempty prompt IDs, response IDs).
Append one EOS, shift input/target by one, and score response plus EOS.
Pad to largest shifted length. Input IDs/targets use pad at padded positions.
Boolean masks; integer IDs. max_length bounds the SHIFTED length.
Reject oversized/empty-prompt/negative-ID examples with ValueError.
Inputs must not be mutated. Empty responses still score EOS."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

A prompt has 5 tokens and its response has 3 tokens, followed by EOS.

- Number of scored targets

<details><summary>Worked answers</summary>

- 4: Three response tokens plus EOS. Prompt length affects context, not this count.

</details>

## Experiment

Run `experiments/scratch_adaptation.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- The experiment learns a tiny instruction-conditioned mapping with held-out prompt variants; log response-only loss and exact-match separately.
- Switch to a prompt-only loss mask as a negative control. Low loss on repeated prompt boilerplate is not successful answer training.

## Transfer Problems

1. Specify a safe truncation policy that preserves at least one response target and the role boundary.
2. Add separate role tokens without permitting user text to create a new trusted role.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[LLMs from Scratch: instruction adaptation](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch07). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Response-only SFT collation. Lists are token IDs, not raw text."""
import numpy as np

def collate_sft(examples, eos, pad, max_length):
    """Return dict input_ids, targets, attention_mask, loss_mask as [B,T] arrays.

examples is a nonempty list of (nonempty prompt IDs, response IDs).
Append one EOS, shift input/target by one, and score response plus EOS.
Pad to largest shifted length. Input IDs/targets use pad at padded positions.
Boolean masks; integer IDs. max_length bounds the SHIFTED length.
Reject oversized/empty-prompt/negative-ID examples with ValueError.
Inputs must not be mutated. Empty responses still score EOS."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
examples = [([10, 11], [20, 21]), ([12], [])]
got = collate_sft(examples, eos=2, pad=0, max_length=4)
assert np.array_equal(got['input_ids'], [[10, 11, 20, 21], [12, 0, 0, 0]])
assert np.array_equal(got['targets'], [[11, 20, 21, 2], [2, 0, 0, 0]])
assert np.array_equal(got['attention_mask'], [[1, 1, 1, 1], [1, 0, 0, 0]])
assert np.array_equal(got['loss_mask'], [[0, 1, 1, 1], [1, 0, 0, 0]]), 'Score first response and EOS, not prompt or padding'
assert got['loss_mask'].dtype.kind == 'b' and got['attention_mask'].dtype.kind == 'b'
assert examples == [([10, 11], [20, 21]), ([12], [])]
for prompt_size in range(1, 5):
    for response_size in range(4):
        result = collate_sft([([9] * prompt_size, [8] * response_size)], 2, 0, 12)
        assert result['loss_mask'].sum() == response_size + 1
        assert result['targets'][result['loss_mask']].tolist() == [8] * response_size + [2]
for ex, length in [([([], [1])], 4), (examples, 3), ([], 4), ([([1], [-1])], 4)]:
    try:
        collate_sft(ex, 2, 0, length)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid/truncated examples')
print('Instruction-collation checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Response-only SFT collation. Lists are token IDs, not raw text."""
import numpy as np


def collate_sft(examples, eos, pad, max_length):
    """Return dict input_ids, targets, attention_mask, loss_mask as [B,T] arrays.

    examples is a nonempty list of (nonempty prompt IDs, response IDs).
    Append one EOS, shift input/target by one, and score response plus EOS.
    Pad to largest shifted length. Input IDs/targets use pad at padded positions.
    Boolean masks; integer IDs. max_length bounds the SHIFTED length.
    Reject oversized/empty-prompt/negative-ID examples with ValueError.
    Inputs must not be mutated. Empty responses still score EOS.
    """
    if not examples or not isinstance(max_length, int) or isinstance(max_length, bool) or max_length < 1:
        raise ValueError('nonempty batch and positive max_length required')
    if any(not isinstance(v, (int, np.integer)) or isinstance(v, (bool, np.bool_)) or v < 0 for v in (eos, pad)):
        raise ValueError('invalid special token ID')
    rows = []
    for prompt, response in examples:
        seq = list(prompt) + list(response) + [eos]
        if not len(prompt) or len(seq) - 1 > max_length or any(not isinstance(v, (int, np.integer)) or isinstance(v, (bool, np.bool_)) or v < 0 for v in seq):
            raise ValueError('invalid or oversized example')
        rows.append((seq, len(prompt)))
    shape = (len(rows), max(len(seq) - 1 for seq, _ in rows))
    x, y = np.full(shape, pad, int), np.full(shape, pad, int)
    attention, loss = np.zeros(shape, bool), np.zeros(shape, bool)
    for b, (seq, prompt_length) in enumerate(rows):
        length = len(seq) - 1
        x[b, :length], y[b, :length] = seq[:-1], seq[1:]
        attention[b, :length] = True
        loss[b, prompt_length - 1:length] = True
    return dict(input_ids=x, targets=y, attention_mask=attention, loss_mask=loss)


In [ ]:
import numpy as np
examples = [([10, 11], [20, 21]), ([12], [])]
got = collate_sft(examples, eos=2, pad=0, max_length=4)
assert np.array_equal(got['input_ids'], [[10, 11, 20, 21], [12, 0, 0, 0]])
assert np.array_equal(got['targets'], [[11, 20, 21, 2], [2, 0, 0, 0]])
assert np.array_equal(got['attention_mask'], [[1, 1, 1, 1], [1, 0, 0, 0]])
assert np.array_equal(got['loss_mask'], [[0, 1, 1, 1], [1, 0, 0, 0]]), 'Score first response and EOS, not prompt or padding'
assert got['loss_mask'].dtype.kind == 'b' and got['attention_mask'].dtype.kind == 'b'
assert examples == [([10, 11], [20, 21]), ([12], [])]
for prompt_size in range(1, 5):
    for response_size in range(4):
        result = collate_sft([([9] * prompt_size, [8] * response_size)], 2, 0, 12)
        assert result['loss_mask'].sum() == response_size + 1
        assert result['targets'][result['loss_mask']].tolist() == [8] * response_size + [2]
for ex, length in [([([], [1])], 4), (examples, 3), ([], 4), ([([1], [-1])], 4)]:
    try:
        collate_sft(ex, 2, 0, length)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid/truncated examples')
print('Instruction-collation checks passed.')


## Existing Course Model

The exact vendored mini_llm configuration and model are embedded here so the notebook does not depend on a repository clone or network import. The Torch model includes learned normalization affine parameters and exact GELU; the browser capstone deliberately omits affine parameters and uses its documented approximation.


In [ ]:
from dataclasses import asdict, dataclass
from typing import Dict, Any


@dataclass
class ModelConfig:
    vocab_size: int
    block_size: int = 64
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1
    bias: bool = False

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if self.vocab_size <= 0 or self.block_size <= 0:
            raise ValueError("vocab_size and block_size must be positive")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

"""A bigram baseline and a small decoder-only Transformer."""
import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
from torch.nn import functional as F

def sample_next_token(logits: torch.Tensor, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError('temperature must be greater than zero')
    logits = logits / temperature
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k).values[:, [-1]]
        logits = logits.masked_fill(logits < threshold, float('-inf'))
    probabilities = F.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1)

class BigramLanguageModel(nn.Module):
    """Each token directly stores logits for the token that follows it."""

    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)
        nn.init.zeros_(self.token_embedding.weight)
        self.vocab_size = vocab_size

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        logits = self.token_embedding(inputs)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        return (logits, loss)

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            (logits, _) = self(tokens[:, -1:])
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

class CausalSelfAttention(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.query_key_value = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.projection = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attention_dropout = nn.Dropout(config.dropout)
        self.residual_dropout = nn.Dropout(config.dropout)
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        (batch_size, time_steps, channels) = inputs.shape
        qkv = self.query_key_value(inputs)
        (query, key, value) = qkv.split(self.n_embd, dim=2)

        def split_heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch_size, time_steps, self.n_head, self.head_dim).transpose(1, 2)
        (query, key, value) = map(split_heads, (query, key, value))
        attention = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = attention.masked_fill(self.causal_mask[:, :, :time_steps, :time_steps] == 0, float('-inf'))
        weights = self.attention_dropout(F.softmax(attention, dim=-1))
        output = weights @ value
        output = output.transpose(1, 2).contiguous().view(batch_size, time_steps, channels)
        return self.residual_dropout(self.projection(output))

class FeedForward(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), nn.GELU(), nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias), nn.Dropout(config.dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)

class TransformerBlock(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)
        self.feed_forward_norm = nn.LayerNorm(config.n_embd)
        self.feed_forward = FeedForward(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        inputs = inputs + self.attention(self.attention_norm(inputs))
        inputs = inputs + self.feed_forward(self.feed_forward_norm(inputs))
        return inputs

class GPT(nn.Module):
    """A compact GPT-2-style decoder-only Transformer."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        (_, time_steps) = inputs.shape
        if time_steps > self.config.block_size:
            raise ValueError(f'Sequence length {time_steps} exceeds block size {self.config.block_size}')
        positions = torch.arange(time_steps, device=inputs.device)
        hidden = self.token_embedding(inputs) + self.position_embedding(positions)
        hidden = self.dropout(hidden)
        for block in self.blocks:
            hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.config.vocab_size), targets.reshape(-1))
        return (logits, loss)

    def parameter_count(self, exclude_position_embeddings: bool=True) -> int:
        count = sum((parameter.numel() for parameter in self.parameters()))
        if exclude_position_embeddings:
            count -= self.position_embedding.weight.numel()
        return count

    def optimizer_groups(self, weight_decay: float) -> list:
        (decay, no_decay) = ([], [])
        for parameter in self.parameters():
            (decay if parameter.dim() >= 2 else no_decay).append(parameter)
        return [{'params': decay, 'weight_decay': weight_decay}, {'params': no_decay, 'weight_decay': 0.0}]

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            context = tokens[:, -self.config.block_size:]
            (logits, _) = self(context)
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

def build_model(model_type: str, config: ModelConfig) -> nn.Module:
    if model_type == 'bigram':
        return BigramLanguageModel(config.vocab_size)
    if model_type == 'gpt':
        return GPT(config)
    raise ValueError("model_type must be 'bigram' or 'gpt'")

def checkpoint_model(checkpoint: Dict[str, object], device: torch.device) -> nn.Module:
    raw_config = checkpoint.get('model_config')
    model_type = checkpoint.get('model_type')
    if not isinstance(raw_config, dict) or not isinstance(model_type, str):
        raise ValueError('Checkpoint is missing model metadata')
    model = build_model(model_type, ModelConfig(**raw_config))
    model.load_state_dict(checkpoint['model_state'])
    return model.to(device)


In [ ]:
"""CPU pretraining, token-weighted accumulation, and exact local resume.

Run: uv run --with numpy --with torch python experiments/scratch_training.py
The notebook embeds the existing mini_llm model source before this cell.
No downloaded data/weights. Fixed synthetic language, not natural-language skill.
"""
import copy
import hashlib
import io
import json
from pathlib import Path
import sys
import torch
from torch.nn import functional as F
if 'GPT' not in globals():
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'colab/vendor/src'))
    from mini_llm.models import GPT
    from mini_llm.config import ModelConfig

def scratch_data(seed=19):
    """Independent unique sequences: next symbol usually advances by one."""
    generator = torch.Generator().manual_seed(seed)
    (rows, seen) = ([], set())
    while len(rows) < 320:
        row = [int(torch.randint(8, (1,), generator=generator))]
        for _ in range(12):
            step = 1 if torch.rand((), generator=generator) < 0.82 else 3
            row.append((row[-1] + step) % 8)
        if tuple(row) not in seen:
            seen.add(tuple(row))
            rows.append(row)
    data = torch.tensor(rows)
    permutation = torch.randperm(len(data), generator=generator)
    data = data[permutation]
    return (data[:224], data[224:272], data[272:])

def scratch_pretrain(seed=7, updates=140):
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    config = ModelConfig(vocab_size=8, block_size=12, n_layer=1, n_head=2, n_embd=24, dropout=0.0)
    model = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.006, weight_decay=0.01)
    (train, validation, test) = scratch_data()
    generator = torch.Generator().manual_seed(seed + 1)

    def evaluate(rows):
        model.eval()
        with torch.no_grad():
            (logits, _) = model(rows[:, :-1])
            return float(F.cross_entropy(logits.flatten(0, 1), rows[:, 1:].flatten()))

    def next_update(m, opt, gen):
        m.train()
        indices = torch.randint(len(train), (24,), generator=gen)
        batch = train[indices]
        opt.zero_grad(set_to_none=True)
        tokens = batch[:, 1:].numel()
        total_loss = 0.0
        for micro in (batch[:7], batch[7:]):
            (logits, _) = m(micro[:, :-1])
            loss_sum = F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum')
            loss = loss_sum / tokens
            loss.backward()
            total_loss += float(loss.detach())
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        return total_loss
    initial = evaluate(validation)
    history = []
    for step in range(updates):
        next_update(model, optimizer, generator)
        if (step + 1) % 35 == 0:
            history.append({'update': step + 1, 'train_nll': evaluate(train), 'validation_nll': evaluate(validation)})
    checkpoint = copy.deepcopy(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=generator.get_state(), torch_rng=torch.get_rng_state(), step=updates, cursor=0, config=config.to_dict(), tokenizer_id='symbols-a-through-h-v1'))
    serialized = io.BytesIO()
    torch.save(checkpoint, serialized)
    serialized.seek(0)
    checkpoint = torch.load(serialized, weights_only=True)
    continued = copy.deepcopy(model)
    continued_optimizer = torch.optim.AdamW(continued.parameters(), lr=0.006, weight_decay=0.01)
    continued_optimizer.load_state_dict(copy.deepcopy(optimizer.state_dict()))
    continued_rng = torch.Generator()
    continued_rng.set_state(generator.get_state())
    with torch.random.fork_rng(devices=[]):
        restored = GPT(ModelConfig(**checkpoint['config']))
        restored.load_state_dict(checkpoint['model'])
        resumed_optimizer = torch.optim.AdamW(restored.parameters(), lr=0.006, weight_decay=0.01)
        resumed_optimizer.load_state_dict(checkpoint['optimizer'])
        resumed_rng = torch.Generator()
        resumed_rng.set_state(checkpoint['rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        uninterrupted = next_update(continued, continued_optimizer, continued_rng)
        torch.set_rng_state(checkpoint['torch_rng'])
        resumed = next_update(restored, resumed_optimizer, resumed_rng)
    assert uninterrupted == resumed
    assert all((torch.equal(a, b) for (a, b) in zip(continued.parameters(), restored.parameters())))
    model.eval()
    batch = train[:9]
    optimizer.zero_grad(set_to_none=True)
    (_, full_loss) = model(batch[:, :-1], batch[:, 1:])
    full_loss.backward()
    full_grads = [p.grad.clone() for p in model.parameters()]
    optimizer.zero_grad(set_to_none=True)
    for micro in (batch[:2], batch[2:]):
        (logits, _) = model(micro[:, :-1])
        (F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum') / batch[:, 1:].numel()).backward()
    error = max((float((p.grad - g).abs().max()) for (p, g) in zip(model.parameters(), full_grads)))
    assert error < 2e-06, error
    final = evaluate(validation)
    assert final < initial, 'This fixed toy run should learn its transition pattern'
    model.eval()
    torch.manual_seed(seed + 100)
    sample = model.generate(torch.tensor([[0]]), 16, top_k=3).tolist()[0]
    report = dict(seed=seed, updates=updates, config=config.to_dict(), parameters=model.parameter_count(False), split_sizes=[len(train), len(validation), len(test)], data_sha256=hashlib.sha256(torch.cat([train, validation, test]).numpy().tobytes()).hexdigest(), initial_validation_nll=initial, final_validation_nll=final, test_nll=evaluate(test), accumulation_max_error=error, resume_exact=True, history=history, generated_ids=sample, tokenizer=dict(symbols=list('abcdefgh'), ids=list(range(8))), generated_text=''.join(('abcdefgh'[i] for i in sample)), limitation='Synthetic transition language; not a general-purpose LLM. Test used once after a fixed recipe.')
    return (model, report)


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Actual classification, response-only SFT, and low-rank adaptation on CPU.

Run: uv run --with numpy --with torch python experiments/scratch_adaptation.py
The notebook embeds scratch_training and the vendor model; local imports reuse it.
"""
import copy
import json
import torch
from torch import nn
from torch.nn import functional as F

if 'scratch_pretrain' not in globals():
    from scratch_training import scratch_pretrain, scratch_data


def adaptation_experiment(seed=11):
    model, pretraining = scratch_pretrain(seed, 70)
    train, validation, test = scratch_data()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    frozen = copy.deepcopy(model.state_dict())

    def features(rows):
        # Use the pretrained decoder's representation, not a new feature model.
        with torch.no_grad():
            ids = rows[:, :-1]
            x = model.token_embedding(ids) + model.position_embedding(torch.arange(ids.shape[1]))
            for block in model.blocks:
                x = block(x)
            return model.final_norm(x)[:, -1]

    model.eval()
    train_x, test_x = features(train), features(test)
    train_y, test_y = (train[:, -2] >= 4).long(), (test[:, -2] >= 4).long()
    head = nn.Linear(train_x.shape[1], 2)
    optimizer = torch.optim.AdamW(head.parameters(), lr=.04)
    for _ in range(100):
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(head(train_x), train_y); loss.backward(); optimizer.step()
    predictions = head(test_x).argmax(-1)
    accuracy = float((predictions == test_y).float().mean())
    baseline = float((test_y == train_y.bincount(minlength=2).argmax()).float().mean())
    f1s = []
    for label in (0, 1):
        tp = int(((predictions == label) & (test_y == label)).sum())
        fp = int(((predictions == label) & (test_y != label)).sum())
        fn = int(((predictions != label) & (test_y == label)).sum())
        f1s.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    assert all(torch.equal(frozen[k], model.state_dict()[k]) for k in frozen)
    assert accuracy > baseline

    # Adapt the pretrained model on a tiny instruction-conditioned task.
    # Templates 0/1 mean copy/successor; nuisance token is split by value.
    # Prompt=[template, nuisance, number]; response=[answer, EOS=7].
    # Numbers 0..3 avoid collision with EOS. Nuisance 4/5 train, 6 holdout.
    adapted = copy.deepcopy(model)
    for p in adapted.parameters():
        p.requires_grad_(True)
    def instruction_rows(nuisances):
        rows = []
        for mode in (0, 1):
            for nuisance in nuisances:
                for number in range(4):
                    rows.append([mode, nuisance, number, (number+mode) % 4, 7])
        return torch.tensor(rows)
    sft_train, sft_test = instruction_rows([4, 5]), instruction_rows([6])
    opt = torch.optim.AdamW(adapted.parameters(), lr=.004)
    def response_loss(rows):
        logits, _ = adapted(rows[:, :-1])
        return F.cross_entropy(logits[:, 2:].reshape(-1, 8), rows[:, 3:].reshape(-1))
    with torch.no_grad():
        initial_sft = float(response_loss(sft_test))
    for _ in range(200):
        opt.zero_grad(set_to_none=True)
        loss = response_loss(sft_train); loss.backward(); opt.step()
    with torch.no_grad():
        final_sft = float(response_loss(sft_test))
        generated = sft_test[:, :3].clone()
        for _ in range(2):
            logits, _ = adapted(generated)
            generated = torch.cat([generated, logits[:, -1].argmax(-1, keepdim=True)], -1)
        exact = float((generated[:, 3:] == sft_test[:, 3:]).all(-1).float().mean())
    assert final_sft < initial_sft

    # Isolated LoRA mechanism: the target differs from a frozen base by rank 2.
    torch.manual_seed(seed)
    x = torch.randn(160, 8)
    base = torch.randn(8, 6)
    base_before = base.clone()
    target = base + torch.randn(8, 2) @ torch.randn(2, 6)
    y = x @ target
    a, b = nn.Parameter(torch.randn(8, 2)*.1), nn.Parameter(torch.zeros(2, 6))
    opt = torch.optim.Adam([a, b], lr=.06)
    initial_lora = float(F.mse_loss(x[120:] @ base, y[120:]))
    for _ in range(250):
        opt.zero_grad(set_to_none=True)
        loss = F.mse_loss(x[:120] @ base + (x[:120] @ a) @ b, y[:120]); loss.backward(); opt.step()
    with torch.no_grad():
        direct = x[120:] @ base + (x[120:] @ a) @ b
        merged = x[120:] @ (base + a @ b)
        merge_error = float((direct-merged).abs().max())
        final_lora = float(F.mse_loss(direct, y[120:]))
    assert torch.equal(base, base_before) and merge_error < 1e-4 and final_lora < initial_lora
    return dict(seed=seed, pretraining_validation_nll=pretraining['final_validation_nll'],
        classification=dict(test_n=len(test), baseline_accuracy=baseline, accuracy=accuracy, macro_f1=sum(f1s)/2, frozen_backbone=True),
        sft=dict(train_n=len(sft_train), test_n=len(sft_test), initial_nll=initial_sft, final_nll=final_sft, exact_match=exact),
        lora=dict(initial_mse=initial_lora, final_mse=final_lora, merge_max_error=merge_error, frozen_base=True),
        limitation='Synthetic tasks test mechanisms. Held-out nuisance values are not evidence of broad instruction generalization.')


if __name__ == '__main__':
    print(json.dumps(adaptation_experiment(), indent=2))
